# LLM Code Assistant with OpenVINO™

This notebook demonstrates how to use **code-specialized Large Language Models** for practical coding tasks, running locally on Intel hardware with [OpenVINO™](https://github.com/openvinotoolkit/openvino).

We use code-specialized models from the [Qwen2.5-Coder](https://huggingface.co/collections/Qwen/qwen25-coder-66eaa22e6f91e9e76f48917d) and [Qwen3-Coder](https://huggingface.co/Qwen/Qwen3-Coder-30B-A3B-Instruct) families with the high-performance [`openvino-genai`](https://github.com/openvinotoolkit/openvino.genai) inference library. Pre-converted OpenVINO models are downloaded from the [OpenVINO collection on HuggingFace](https://huggingface.co/OpenVINO) when available; otherwise, models are converted locally using [Optimum Intel](https://huggingface.co/docs/optimum/intel/index).

**Demonstrations:**
1. **Bug Detection & Code Correction** — fix multiple bugs with automatic test verification.
2. **Interactive Code Generation** — write a functional Snake game from a short prompt.
3. **Security Audit Agent** — agentic workflow: find vulnerabilities, fix them, self-correct on errors.
4. **Codebase Explorer Agent** — agentic workflow: model explores a GitHub repo, decides what files to fetch, builds comprehensive analysis.
5. **Interactive Chat** — pair-program via Gradio.

Each demo reports **generation speed (tokens/s)** so you can see OpenVINO performance in action.

#### Table of contents:

- [Installation](#Installation)
- [Configuration](#Configuration)
- [Convert and Download Model](#Convert-and-Download-Model)
- [Run Inference with OpenVINO GenAI](#Run-Inference-with-OpenVINO-GenAI)
- [Demo 1: Bug Detection and Code Correction](#Demo-1:-Bug-Detection-and-Code-Correction)
- [Demo 2: Interactive Web App Generation](#Demo-2:-Interactive-Web-App-Generation)
- [Demo 3: Security Audit Agent](#Demo-3:-Security-Audit-Agent)
- [Demo 4: Codebase Explorer Agent](#Demo-4:-Codebase-Explorer-Agent)
- [Interactive Assistant with Gradio](#Interactive-Assistant-with-Gradio)
- [Cleanup](#Cleanup)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-code-assistant/llm-code-assistant.ipynb" />

## Installation
[back to top ⬆️](#Table-of-contents:)

Install the required libraries, including `openvino-genai` for optimal performance.

In [ ]:
import os
import platform
import shutil
import requests
from pathlib import Path

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

if not Path("pip_helper.py").exists():
    pip_helper_shared = Path("../../utils/pip_helper.py")
    if pip_helper_shared.exists():

        shutil.copy(pip_helper_shared, "pip_helper.py")
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py")
        open("pip_helper.py", "w").write(r.text)

from pip_helper import pip_install

pip_install("-q", "-U", "openvino>=2026.0.0", "openvino-genai>=2026.0.0")
pip_install(
    "-q",
    "-U",
    "gradio>=6.0.0",
    "huggingface_hub",
    "git+https://github.com/huggingface/optimum-intel.git",
    "nncf>=2.18.0",
    "torch==2.8",
    "datasets<4.0.0",
    "accelerate",
    "transformers>=4.55",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)

if platform.system() == "Darwin":
    pip_install("-q", "numpy<2.0.0")

In [ ]:
# Fetch llm_config.py (model definitions and conversion helpers)
config_shared_path = Path("../../utils/llm_config.py")
config_dst_path = Path("llm_config.py")

if not config_dst_path.exists():
    if config_shared_path.exists():
        try:
            os.symlink(config_shared_path, config_dst_path)
        except Exception:
            shutil.copy(config_shared_path, config_dst_path)
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/llm_config.py")
        with open("llm_config.py", "w", encoding="utf-8") as f:
            f.write(r.text)
elif not os.path.islink(config_dst_path):
    if config_shared_path.exists():
        shutil.copy(config_shared_path, config_dst_path)
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/llm_config.py")
        with open("llm_config.py", "w", encoding="utf-8") as f:
            f.write(r.text)

# Fetch notebook_utils.py
if not Path("notebook_utils.py").exists():
    notebook_utils_shared = Path("../../utils/notebook_utils.py")
    if notebook_utils_shared.exists():
        try:
            os.symlink(notebook_utils_shared, "notebook_utils.py")
        except Exception:
            shutil.copy(notebook_utils_shared, "notebook_utils.py")
    else:
        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
        open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-code-assistant.ipynb")

## Configuration
[back to top ⬆️](#Table-of-contents:)

Select the model, compression level, and target device. The notebook supports the full lineup of code-specialized models:

- **Qwen2.5-Coder** family: 7B, 14B — strong general-purpose code models.
- **Qwen3-Coder-30B-A3B-Instruct** — MoE architecture (30B total, 3.3B active per token).

Pre-converted models are downloaded from the [OpenVINO collection on HuggingFace](https://huggingface.co/OpenVINO) when available. If a pre-converted model is not found, the notebook converts it locally using [Optimum Intel](https://huggingface.co/docs/optimum/intel/index).

In [3]:
import ipywidgets as widgets
from notebook_utils import device_widget
from llm_config import SUPPORTED_LLM_MODELS

# Filter to code-specific models only
CODE_MODEL_KEYS = [
    "Qwen3-Coder-30B-A3B-Instruct",
    "qwen2.5-coder-7b-instruct",
    "qwen2.5-coder-14b-instruct",
]

all_models = SUPPORTED_LLM_MODELS["English"]
code_models = {k: v for k, v in all_models.items() if k in CODE_MODEL_KEYS}

model_dropdown = widgets.Dropdown(
    options=code_models,
    description="Model:",
)

compression_dropdown = widgets.Dropdown(
    options=["INT4", "INT8", "FP16"],
    value="INT4",
    description="Compression:",
)

use_preconverted = widgets.Checkbox(value=True, description="Use preconverted")

device_dropdown = device_widget(default="AUTO")

display(model_dropdown, compression_dropdown, use_preconverted, device_dropdown)

Dropdown(description='Model:', options={'Qwen3-Coder-30B-A3B-Instruct': {'model_id': 'Qwen/Qwen3-Coder-30B-A3B…

Dropdown(description='Compression:', options=('INT4', 'INT8', 'FP16'), value='INT4')

Checkbox(value=True, description='Use preconverted')

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

## Convert and Download Model
[back to top ⬆️](#Table-of-contents:)

Download the pre-converted OpenVINO model from HuggingFace Hub, or convert it locally if a pre-converted version is not available.

In [4]:
from llm_config import convert_and_compress_model

model_configuration = model_dropdown.value
model_id = model_dropdown.label
precision = compression_dropdown.value

print(f"Selected model: {model_id} with {precision} compression")

model_path = convert_and_compress_model(model_id, model_configuration, precision, use_preconverted.value)

Selected model: Qwen3-Coder-30B-A3B-Instruct with INT4 compression
⌛Found preconverted INT4 Qwen3-Coder-30B-A3B-Instruct. Downloading model started. It may takes some time.


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

chat_template_original.jinja: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ INT4 Qwen3-Coder-30B-A3B-Instruct model downloaded and can be found in Qwen3-Coder-30B-A3B-Instruct/INT4_compressed_weights


## Run Inference with OpenVINO GenAI
[back to top ⬆️](#Table-of-contents:)

Load the model and define the generation helper function.

In [5]:
import time
import openvino_genai as ov_genai

device = device_dropdown.value

print(f"Loading model on {device}...")
pipe = ov_genai.LLMPipeline(str(model_path), device)
print(f"Model loaded successfully on {device}!")

DEFAULT_SYSTEM_MESSAGE: str = "You are an expert programming assistant. Write clean, efficient code."


def generate(user_text: str, max_tokens: int = 512) -> str:
    """Generate text from a user message. When start_chat() is active, the pipeline
    applies the chat template automatically. Otherwise user_text is raw input.

    :param user_text: user message or raw prompt
    :param max_tokens: maximum number of tokens to generate
    :returns: generated text
    """
    start = time.perf_counter()
    result = pipe.generate(user_text, max_new_tokens=max_tokens, do_sample=True, temperature=0.2, top_p=0.95)
    elapsed = time.perf_counter() - start
    text = str(result)

    # Use API performance metrics when available, fall back to word-count estimation
    if hasattr(result, "perf_metrics"):
        perf = result.perf_metrics
        num_tokens = perf.get_num_generated_tokens()
        throughput = perf.get_throughput().mean
        print(f"⚡ Generated {num_tokens} tokens in {elapsed:.1f}s ({throughput:.1f} tokens/s on {device})")
    else:
        approx_tokens = int(len(text.split()) / 0.75)
        tokens_per_sec = approx_tokens / elapsed if elapsed > 0 else 0
        print(f"⚡ Generated ~{approx_tokens} tokens in {elapsed:.1f}s ({tokens_per_sec:.1f} tokens/s on {device})")

    return text


def generate_code(prompt: str, max_tokens: int = 512, system_message: str = DEFAULT_SYSTEM_MESSAGE) -> str:
    """Single-turn code generation with system message via chat API.

    Uses start_chat()/finish_chat() so the pipeline applies the correct
    chat template automatically — no model-specific markup needed.

    :param prompt: user message
    :param max_tokens: maximum number of tokens to generate
    :param system_message: system message for the chat session
    :returns: generated text
    """
    pipe.start_chat(system_message=system_message)
    try:
        return generate(prompt, max_tokens=max_tokens)
    finally:
        pipe.finish_chat()

Loading model on CPU...
Model loaded successfully on CPU!


## Demo 1: Bug Detection and Code Correction
[back to top ⬆️](#Table-of-contents:)

The model receives Python functions with **multiple bugs** and must fix them all. We verify the fix by running corrected functions against test cases.

In [ ]:
import ast
import re

broken_code = """
def merge_sorted_lists(list1, list2):
    \"\"\"Merge two sorted lists into one sorted list.\"\"\"
    result = []
    i = j = 0
    while i <= len(list1) and j <= len(list2):  # Bug 1: should be <, not <=
        if list1[i] < list2[j]:
            result.append(list1[i])
            i += 1
        else:
            result.append(list2[j])
            j += 1
    result += list1[i:]
    result += list2[j:]
    return result

def find_duplicates(lst):
    \"\"\"Return a list of duplicate elements.\"\"\"
    seen = set()
    duplicates = []
    for item in lst:
        if item in seen:
            duplicates.append(item)  # Bug 2: appends every time, not unique duplicates
        seen.add(item)
    return duplicates

def flatten(nested_list):
    \"\"\"Flatten a nested list.\"\"\"
    result = []
    for item in nested_list:
        if type(item) == list:  # Bug 3: doesn't handle tuples/other iterables
            result.extend(item)  # Bug 4: extend instead of recursive flatten
        else:
            result.append(item)
    return result
"""

prompt = f"""Fix ALL bugs in the following Python code. For each bug:
1. State what the bug is
2. Show the fix

Then provide the complete corrected code in a ```python block.

```python
{broken_code}
```"""

print("Analyzing code for bugs...\n")
response = generate_code(prompt, max_tokens=1024)

# Remove thinking blocks
response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
print(response)

# Extract and verify syntax of the fixed code
fixed_code = None
if "```python" in response:
    fixed_code = response.split("```python")[1].split("```")[0].strip()
elif "```" in response:
    fixed_code = response.split("```")[1].split("```")[0].strip()

if fixed_code:
    try:
        ast.parse(fixed_code)
        print("\n✅ Extracted code is syntactically valid (AST check passed).")
    except SyntaxError as e:
        print(f"\n❌ Fixed code has syntax error: {e}")
        fixed_code = None
else:
    print("\n⚠ Could not extract valid code from model response.")

🔍 Analyzing code for bugs...

⚡ Generated ~376 tokens in 37.0s (10.2 tokens/s on CPU)
Let me analyze each function for bugs:

## Bug Analysis:

**Function 1: `merge_sorted_lists`**
1. **Bug**: Using `<=` instead of `<` in the while condition
   - **Problem**: This will cause index out of bounds errors when `i` or `j` equals the length of their respective lists
   - **Fix**: Change `<=` to `<`

**Function 2: `find_duplicates`**
2. **Bug**: Appends duplicate items every time they're encountered
   - **Problem**: The function returns multiple copies of the same duplicate element
   - **Fix**: Only append once per duplicate element, or use a set to track seen duplicates

**Function 3: `flatten`**
3. **Bug**: Only handles direct list nesting, not deeply nested structures
   - **Problem**: Doesn't recursively flatten nested lists
   - **Fix**: Make it recursive to handle arbitrary nesting levels

4. **Bug**: Only checks for `list` type, ignores other iterable types
   - **Problem**: Tuples a

**⚠ Security note:** The cell below can **optionally execute LLM-generated code** to verify correctness against test cases. Execution is **disabled by default** — set `ENABLE_EXEC_VERIFICATION = True` to enable it.

Before execution, an AST-based safety check rejects code containing imports, file/OS/network access, or dangerous builtins (`exec`, `eval`, `open`, `__import__`, `os.system`, etc.). The code runs in an isolated namespace with no access to the notebook's variables.

> **If you reuse this pattern in your own project**, always keep the `is_safe_for_exec()` guard — never run `exec()` on LLM output without static analysis first.

In [ ]:
# Set to True to execute the LLM-generated fix and run test cases.
# When False (default), the fix is only validated via AST parsing — no code is executed.
ENABLE_EXEC_VERIFICATION = False


def is_safe_for_exec(code_str: str) -> tuple[bool, str]:
    """AST-based safety check: reject code with imports, file/OS/network access, or dangerous builtins.

    :param code_str: Python source code to validate
    :returns: (is_safe, reason) — reason is empty string when safe
    """
    tree = ast.parse(code_str)
    dangerous_names = {"exec", "eval", "compile", "open", "__import__", "getattr", "setattr", "delattr", "globals", "locals", "breakpoint"}
    dangerous_attrs = {"system", "popen", "exec", "eval", "remove", "rmdir", "unlink"}
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return False, f"Code contains import: {ast.dump(node)}"
        if isinstance(node, ast.Call):
            func = node.func
            if isinstance(func, ast.Name) and func.id in dangerous_names:
                return False, f"Code calls dangerous builtin: {func.id}"
            if isinstance(func, ast.Attribute) and func.attr.lower() in dangerous_attrs:
                return False, f"Code calls dangerous method: {func.attr}"
    return True, ""


if fixed_code:
    if not ENABLE_EXEC_VERIFICATION:
        print("ℹ Execution verification is disabled (ENABLE_EXEC_VERIFICATION = False).")
        print("  The fixed code passed AST syntax check above. Set the flag to True and re-run to execute test cases.")
    else:
        is_safe, safety_reason = is_safe_for_exec(fixed_code)
        if not is_safe:
            print(f"⚠ Skipping execution — generated code failed safety check: {safety_reason}")
            print("The fixed code is shown above for manual review.")
        else:
            test_ns: dict = {}
            exec(fixed_code, test_ns)  # noqa: S102 — guarded by is_safe_for_exec() AST check above

            tests = [
                ("merge_sorted_lists([1,3,5], [2,4,6])", [1, 2, 3, 4, 5, 6]),
                ("merge_sorted_lists([], [1,2,3])", [1, 2, 3]),
                ("find_duplicates([1,2,2,3,3,3,4])", [2, 3]),
                ("flatten([1,[2,3],[4,[5,6]],7])", [1, 2, 3, 4, 5, 6, 7]),
            ]

            print("=" * 50)
            print("Verification — running fixed code against test cases:")
            print("=" * 50)
            all_passed = True
            for expr, expected in tests:
                result = eval(expr, test_ns)  # noqa: S307 — test_ns is an isolated namespace, expr is a hardcoded test string
                if "duplicates" in expr:
                    passed = set(result) == set(expected)
                else:
                    passed = result == expected
                status = "✅" if passed else "❌"
                if not passed:
                    all_passed = False
                print(f"  {status} {expr}")
                print(f"       Expected: {expected}")
                print(f"       Got:      {result}")

            print("=" * 50)
            if all_passed:
                print("✅ All tests passed — model fixed the bugs correctly!")
            else:
                print("⚠ Some tests failed — model's fix is incomplete.")
else:
    print("⚠ No valid fixed code to test. Re-run the previous cell.")

🧪 Verification — running fixed code against test cases:
  ✅ merge_sorted_lists([1,3,5], [2,4,6])
       Expected: [1, 2, 3, 4, 5, 6]
       Got:      [1, 2, 3, 4, 5, 6]
  ✅ merge_sorted_lists([], [1,2,3])
       Expected: [1, 2, 3]
       Got:      [1, 2, 3]
  ✅ find_duplicates([1,2,2,3,3,3,4])
       Expected: [2, 3]
       Got:      [2, 3]
  ✅ flatten([1,[2,3],[4,[5,6]],7])
       Expected: [1, 2, 3, 4, 5, 6, 7]
       Got:      [1, 2, 3, 4, 5, 6, 7]
✅ All tests passed — model fixed the bugs correctly!


## Demo 2: Interactive Web App Generation
[back to top ⬆️](#Table-of-contents:)

The model creates a functional HTML/JS game that you can play directly in this notebook.

In [ ]:
import html as html_module

import re
from IPython.display import display, HTML

game_prompt: str = """
Write a complete, single-file HTML5 Snake game. Return ONLY the HTML code, no explanations.
Requirements:
- Canvas 400x400, dark theme, green snake, red food
- Arrow keys to move, snake wraps around edges
- Score counter, pause on Space/Escape
- Game over on self-collision, restart on any key press
- Do NOT use alert(), confirm() or prompt() dialogs
- Start with <!DOCTYPE html>
"""

print("Generating Snake Game... (This may take a moment)")
game_code: str = generate_code(game_prompt, max_tokens=2048)

# Remove <think>...</think> blocks (model thinking mode)
game_code = re.sub(r"<think>.*?</think>", "", game_code, flags=re.DOTALL).strip()

# Extract HTML from markdown code blocks if present
if "```html" in game_code:
    game_code = game_code.split("```html")[1].split("```")[0].strip()
elif "```" in game_code:
    game_code = game_code.split("```")[1].split("```")[0].strip()

try:
    # Always show the generated code first
    print("=" * 60)
    print("Generated HTML code:")
    print("=" * 60)
    print(game_code.encode("ascii", errors="replace").decode("ascii"))
    print("=" * 60)

    # Validate: must contain key elements to be a working game
    is_valid: bool = all(tag in game_code.lower() for tag in ["<canvas", "setinterval", "keydown"])

    if not is_valid:
        print("\n⚠ Generated code doesn't look like a complete game. Try re-running this cell.")
    else:
        game_code_escaped: str = html_module.escape(game_code, quote=True)
        # sandbox="allow-scripts" blocks alert/confirm/prompt dialogs while allowing JS to run
        # Wrapper with a close button to dismiss the game at any time
        display(
            HTML(
                f"""
        <div id="game-wrapper" style="position:relative; display:inline-block; margin-top:10px;">
            <button onclick="document.getElementById('game-wrapper').remove()"
                    style="position:absolute; top:-10px; right:-10px; z-index:10;
                           background:#f85149; color:#fff; border:none; border-radius:50%;
                           width:28px; height:28px; font-size:16px; cursor:pointer;
                           font-weight:bold; line-height:28px; text-align:center;"
                    title="Close game">✕</button>
            <iframe sandbox="allow-scripts" srcdoc="{game_code_escaped}"
                    width="420" height="420" style="border:2px solid #333; display:block;"></iframe>
        </div>
        """
            )
        )
        print("Arrow keys -- move | Space/Esc -- pause | X button -- close game")
except Exception as e:
    print(f"Demo 2 display error (non-critical): {e}")

Generating Snake Game... (This may take a moment)
⚡ Generated ~680 tokens in 75.0s (9.1 tokens/s on CPU)
📝 Generated HTML code:
<!DOCTYPE html>
<html>
<head>
    <title>Snake Game</title>
    <style>
        body {
            background-color: #1a1a1a;
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
            margin: 0;
            font-family: Arial, sans-serif;
        }
        canvas {
            border: 2px solid #333;
            background-color: #000;
        }
        #gameOver {
            position: absolute;
            color: white;
            text-align: center;
            display: none;
        }
        #score {
            position: absolute;
            top: 10px;
            left: 10px;
            color: white;
        }
    </style>
</head>
<body>
    <div id="score">Score: 0</div>
    <canvas id="gameCanvas" width="400" height="400"></canvas>
    <div id="gameOver">Game Over!<br>Pres

🎮 Arrow keys — move | Space/Esc — pause | ✕ button — close game


## Demo 3: Security Audit Agent
[back to top ⬆️](#Table-of-contents:)

This demo showcases the model as a **security-focused code agent** — a practical, high-value use case for AI coding assistants.

**Agentic workflow:**
1. **Analyze**: The model receives Python code with intentional security vulnerabilities.
2. **Report**: It identifies each vulnerability with severity and CWE classification.
3. **Fix**: It generates a corrected version of the code.
4. **Verify**: We validate the fix via AST parsing (safe, no code execution).
5. **Retry**: If the fix has syntax errors, the error is fed back for correction.

> The vulnerable code contains real-world issues: **SQL injection**, **hardcoded credentials**, **path traversal**, and **command injection** — common patterns from the OWASP Top 10.

In [ ]:
import html as html_module
import re
import ast
from IPython.display import display, HTML

# ---- Vulnerable code sample (intentionally insecure for demonstration) ----
VULNERABLE_CODE: str = '''
import os
import sqlite3

# Hardcoded credentials (CWE-798)
DB_PASSWORD = "admin123"
API_KEY = "sk-proj-abc123secret456"

def get_user(username):
    """Fetch user from database."""
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    # SQL Injection (CWE-89)
    query = f"SELECT * FROM users WHERE username = \'{username}\'"
    cursor.execute(query)
    return cursor.fetchone()

def read_file(filename):
    """Read a file from the reports directory."""
    # Path Traversal (CWE-22)
    filepath = f"/var/reports/{filename}"
    with open(filepath, "r") as f:
        return f.read()

def run_diagnostic(host):
    """Ping a host to check connectivity."""
    # Command Injection (CWE-78)
    result = os.system(f"ping -c 1 {host}")
    return result
'''


def verify_syntax(code_string: str) -> tuple[bool, str | None]:
    """Verify code is syntactically valid using AST parsing (no execution).

    :param code_string: Python source code to check
    :returns: tuple of (is_valid, error_message_or_none)
    """
    try:
        ast.parse(code_string)
        return True, None
    except SyntaxError as e:
        return False, f"SyntaxError at line {e.lineno}: {e.msg}"


def extract_sections(response: str) -> tuple[str, str | None]:
    """Extract the security report and fixed code from model response.

    :param response: raw model response text
    :returns: tuple of (report_text, fixed_code_or_none)
    """
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()

    fixed_code = None
    if "```python" in response:
        fixed_code = response.split("```python")[1].split("```")[0].strip()
    elif "```" in response:
        fixed_code = response.split("```")[1].split("```")[0].strip()

    report = response.split("```")[0].strip() if "```" in response else response
    return report, fixed_code


def format_output(vulnerable: str, report: str, fixed_code: str | None) -> str:
    """Format results as styled HTML for clear visual comparison.

    :param vulnerable: original vulnerable source code
    :param report: security audit report text
    :param fixed_code: corrected source code or None
    :returns: HTML string for display
    """
    esc_vulnerable = html_module.escape(vulnerable.strip())
    esc_report = html_module.escape(report)
    esc_fixed = html_module.escape(fixed_code) if fixed_code else "No fixed code generated."

    return f"""
    <style>
        .audit-container {{ font-family: 'Consolas', 'Monaco', monospace; margin: 10px 0; }}
        .audit-section {{ border-radius: 8px; padding: 15px; margin: 10px 0; }}
        .vuln-section {{ background: #2d1117; border: 1px solid #f85149; }}
        .report-section {{ background: #1a1a2e; border: 1px solid #58a6ff; }}
        .fixed-section {{ background: #0d1117; border: 1px solid #3fb950; }}
        .section-title {{ font-size: 14px; font-weight: bold; margin-bottom: 10px; }}
        .vuln-title {{ color: #f85149; }}
        .report-title {{ color: #58a6ff; }}
        .fixed-title {{ color: #3fb950; }}
        pre.code-block {{ background: #161b22; color: #e6edf3; padding: 12px;
                         border-radius: 6px; overflow-x: auto; font-size: 12px;
                         line-height: 1.5; white-space: pre-wrap; }}
        .report-text {{ color: #c9d1d9; font-size: 13px; line-height: 1.6; white-space: pre-wrap; }}
    </style>
    <div class="audit-container">
        <div class="audit-section vuln-section">
            <div class="section-title vuln-title">VULNERABLE CODE (input)</div>
            <pre class="code-block">{esc_vulnerable}</pre>
        </div>
        <div class="audit-section report-section">
            <div class="section-title report-title">SECURITY AUDIT REPORT</div>
            <div class="report-text">{esc_report}</div>
        </div>
        <div class="audit-section fixed-section">
            <div class="section-title fixed-title">FIXED CODE (output)</div>
            <pre class="code-block">{esc_fixed}</pre>
        </div>
    </div>"""


def run_security_agent(vulnerable_code: str, max_attempts: int = 3) -> None:
    """Security audit agent with self-correction loop.

    :param vulnerable_code: Python source code with intentional vulnerabilities
    :param max_attempts: maximum number of correction attempts
    """
    print("Security Audit Agent starting...\n")

    syntax_errors: list[tuple[str, str]] = []

    for attempt in range(1, max_attempts + 1):
        print(f"{'='*50}")
        print(f"Attempt {attempt}/{max_attempts}")
        print(f"{'='*50}")

        if not syntax_errors:
            user_msg = f"Analyze this code for security vulnerabilities and provide the fixed version:\n\n```python\n{vulnerable_code}\n```"
            system_msg = (
                "You are a security code auditor. Analyze the Python code for security vulnerabilities.\n\n"
                "Your response MUST have two parts:\n"
                "1. A vulnerability report listing each issue with: vulnerability name, CWE ID, severity, and brief explanation.\n"
                "2. The complete fixed code in a ```python block with all vulnerabilities resolved.\n\n"
                "Use parameterized queries, environment variables, path validation, and input sanitization."
            )
        else:
            last_code, last_error = syntax_errors[-1]
            system_msg = "You are a security code auditor. Your previous fixed code had a syntax error. Provide the corrected version in a ```python block."
            user_msg = (
                f"Original vulnerable code:\n```python\n{vulnerable_code}\n```\n\n"
                f"Your previous fix (has syntax error):\n```python\n{last_code}\n```\n\n"
                f"Syntax error: {last_error}\n\nFix the syntax error while keeping all security improvements."
            )

        response = generate_code(user_msg, max_tokens=1024, system_message=system_msg)

        report, fixed_code = extract_sections(response)

        if not fixed_code:
            print("⚠ Model did not return a code block.")
            print(f"Raw response:\n{response[:500]}...")
            continue

        is_valid, syntax_error = verify_syntax(fixed_code)

        if is_valid:
            print("✅ Fixed code passed syntax verification (AST parse).")
            if attempt > 1:
                print(f"Model self-corrected after {attempt - 1} syntax error(s).\n")
            display(HTML(format_output(vulnerable_code, report, fixed_code)))
            return
        else:
            print(f"❌ Fix has syntax error: {syntax_error}")
            syntax_errors.append((fixed_code, syntax_error))
            if attempt < max_attempts:
                print("   → Feeding error back to model...\n")

    print(f"\n⚠ Agent could not produce valid fix after {max_attempts} attempts.")


run_security_agent(VULNERABLE_CODE)

🔒 Security Audit Agent starting...

🔄 Attempt 1/3
⚡ Generated ~648 tokens in 59.0s (11.0 tokens/s on CPU)
✅ Fixed code passed syntax verification (AST parse).


## Demo 4: Codebase Explorer Agent
[back to top ⬆️](#Table-of-contents:)

An **agentic** code exploration workflow: the model autonomously decides what context it needs to understand unfamiliar code.

**Agentic loop:**
1. **Explore**: The model receives a single file from GitHub and performs initial analysis.
2. **Request**: It decides which additional files it needs and outputs their paths.
3. **Fetch**: We automatically retrieve the requested files from GitHub.
4. **Synthesize**: The model receives the full context and produces a comprehensive analysis.

The model drives the exploration — it chooses what to look at next, not us.

> **Meta-twist:** OpenVINO GenAI analyzes its own source code — we start with [chat_sample.py](https://github.com/openvinotoolkit/openvino.genai/blob/master/samples/python/text_generation/chat_sample.py) from the `openvino.genai` repository.

In [ ]:
import json
import re
import urllib.error
import urllib.request

REPO_BASE_URL: str = "https://raw.githubusercontent.com/openvinotoolkit/openvino.genai/refs/heads/master"
INITIAL_FILE: str = "samples/python/text_generation/chat_sample.py"


def fetch_github_file(file_path: str) -> str | None:
    """Fetch a file from the openvino.genai GitHub repository.

    :param file_path: path relative to the repository root
    :returns: file content as string, or None if not found / on error
    """
    url = f"{REPO_BASE_URL}/{file_path}"
    try:
        # noqa: S310 — URL is safe: base is a hardcoded GitHub raw content URL (REPO_BASE_URL constant),
        # and file_path is validated by is_safe_repo_path() before calling this function
        # (no traversal, no schemes, no absolute paths). No user-controlled URL components.
        return urllib.request.urlopen(url, timeout=15).read().decode()  # noqa: S310
    except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError):
        return None


def is_safe_repo_path(file_path: str) -> bool:
    """Validate that a model-requested path is safe (no traversal, no schemes, no absolute paths).

    :param file_path: file path to validate
    :returns: True if safe
    """
    if not isinstance(file_path, str):
        return False
    return not (".." in file_path or file_path.startswith("/") or "://" in file_path or "\\" in file_path or file_path.startswith("~"))


# ---- Step 1: Fetch the initial file ----
print(f"Fetching {INITIAL_FILE} from GitHub...\n")
initial_code: str | None = fetch_github_file(INITIAL_FILE)
if not initial_code:
    raise RuntimeError(f"Failed to fetch {INITIAL_FILE}")
print(f"✅ Fetched {len(initial_code)} bytes ({len(initial_code.splitlines())} lines)\n")
print(initial_code)

# ---- Step 2: Model decides what additional context it needs ----
explore_prompt: str = f"""You are exploring the openvino.genai repository (https://github.com/openvinotoolkit/openvino.genai).

Here is the file `{INITIAL_FILE}`:

```python
{initial_code}
```

To fully understand how this code works, you need to see related files from the same repository.

Provide:
1. A brief initial analysis (2-3 sentences about what this file does).
2. A JSON array of up to 3 file paths you want to examine next.
   Format: ```json
["path/one.py", "path/two.py"]
```

Known paths in this repo:
- samples/python/text_generation/README.md
- samples/python/text_generation/greedy_causal_lm.py
- samples/python/text_generation/multinomial_causal_lm.py
- src/python/openvino_genai/py_openvino_genai.pyi"""

print("=" * 60)
print("Step 2: Initial analysis -- model decides what files to explore")
print("=" * 60)
explore_response: str = generate_code(explore_prompt, max_tokens=512)
explore_response = re.sub(r"<think>.*?</think>", "", explore_response, flags=re.DOTALL).strip()
print(explore_response)

# Extract JSON file list from model response
requested_files: list[str] = []
json_match = re.search(r"```json\s*\n(\[.*?\])\s*\n```", explore_response, re.DOTALL)
if json_match:
    try:
        requested_files = json.loads(json_match.group(1))
    except json.JSONDecodeError:
        pass

if not requested_files:
    array_match = re.search(r'\[(?:\s*"[^"]+"\s*,?\s*)+\]', explore_response)
    if array_match:
        try:
            requested_files = json.loads(array_match.group(0))
        except json.JSONDecodeError:
            pass

# ---- Step 3: Fetch files requested by the model ----
print(f"\n{'=' * 60}")
print(f"Step 3: Fetching {len(requested_files)} file(s) requested by model")
print(f"{'=' * 60}")

additional_context: str = ""
fetched_count: int = 0

for file_path in requested_files[:3]:
    if not is_safe_repo_path(file_path):
        print(f"  ⚠ Skipping suspicious path: {file_path}")
        continue

    print(f"  {file_path}...", end=" ")
    content = fetch_github_file(file_path)

    if content:
        if len(content) > 5000:
            content = content[:5000] + "\n# ... (truncated to 5000 chars)"
        additional_context += f"\n\n--- {file_path} ---\n```\n{content}\n```"
        fetched_count += 1
        print(f"✅ ({len(content.splitlines())} lines)")
    else:
        additional_context += f"\n\n--- {file_path} ---\n(File not found in repository)"
        print("❌ not found")

print(f"\n  Fetched {fetched_count}/{len(requested_files)} requested files.")

# ---- Step 4: Final comprehensive analysis with full context ----
print(f"\n{'=' * 60}")
print("Step 4: Comprehensive analysis with gathered context")
print("=" * 60)

final_prompt: str = f"""You previously analyzed `{INITIAL_FILE}` and requested additional files.

**Original file** (`{INITIAL_FILE}`):
```python
{initial_code}
```

**Additional files you requested:**
{additional_context if additional_context else "(No additional files were fetched)"}

Now provide a comprehensive analysis:
1. **Architecture overview** — how the components work together, data flow, key abstractions.
2. **API insights** — what you learned from the additional files about the underlying openvino_genai API (ChatHistory, GenerationConfig, streaming, etc.).
3. **Extension example** — show a modified version of chat_sample.py that adds system prompt support and temperature control via GenerationConfig."""

final_response: str = generate_code(final_prompt, max_tokens=2048)
final_response = re.sub(r"<think>.*?</think>", "", final_response, flags=re.DOTALL).strip()
print(final_response)

📥 Fetching samples/python/text_generation/chat_sample.py from GitHub...

✅ Fetched 1245 bytes (40 lines)

#!/usr/bin/env python3
# Copyright (C) 2024 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

import argparse
import openvino_genai


def streamer(subword):
    print(subword, end="", flush=True)
    # Return flag corresponds whether generation should be stopped.
    return openvino_genai.StreamingStatus.RUNNING


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("model_dir", help="Path to the model directory")
    parser.add_argument("device", nargs="?", default="CPU", help="Device to run the model on (default: CPU)")
    args = parser.parse_args()

    device = args.device
    pipe = openvino_genai.LLMPipeline(args.model_dir, device)

    config = openvino_genai.GenerationConfig()
    config.max_new_tokens = 100

    chat_history = openvino_genai.ChatHistory()
    while True:
        try:
            prompt = input("question:\n")
        except EOF

## Interactive Assistant with Gradio
[back to top ⬆️](#Table-of-contents:)

Launch a local chat interface to pair-program with the code assistant.

In [ ]:
import gradio as gr
import threading
from queue import Queue

# Guard against concurrent access to the shared pipeline (single KV-cache session)
_generate_lock = threading.Lock()


def chat_function(message: str, history: list[list[str]]):
    """Handle chat messages with streaming via chat API.

    On the first message (empty history) we start a new chat session.
    Subsequent messages reuse the pipeline's KV cache for efficient multi-turn dialogue.
    Uses a streamer callback for real-time token-by-token output in the Gradio UI.

    :param message: current user message
    :param history: list of [user_msg, bot_msg] pairs
    :yields: partial response text, updated after each generated token
    """
    q: Queue[str | None] = Queue()

    def streamer(subword: str) -> ov_genai.StreamingStatus:
        q.put(subword)
        return ov_genai.StreamingStatus.RUNNING

    def run():
        with _generate_lock:
            try:
                if not history:
                    pipe.start_chat(system_message=DEFAULT_SYSTEM_MESSAGE)
                pipe.generate(message, max_new_tokens=512, do_sample=True, temperature=0.2, top_p=0.95, streamer=streamer)
            finally:
                q.put(None)  # always signal end of generation

    thread = threading.Thread(target=run, daemon=True)
    thread.start()

    partial_text = ""
    while True:
        token = q.get()
        if token is None:
            break
        partial_text += token
        yield partial_text

    thread.join()


def on_clear():
    """End the chat session when the user clears the conversation."""
    pipe.finish_chat()


with gr.Blocks(fill_height=True) as demo:
    chat = gr.ChatInterface(
        fn=chat_function,
        title="LLM Code Assistant + OpenVINO™",
        description="Ask me to write code, debug programs, or explain concepts. Running locally on your hardware!",
        examples=[
            "Write a binary search function in Python",
            "Explain the difference between threading and multiprocessing",
            "Write a C++ quicksort implementation",
        ],
    )
    chat.chatbot.clear(on_clear)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/

## Cleanup
[back to top ⬆️](#Table-of-contents:)

Remove the downloaded model to free disk space.

In [ ]:
# import shutil

# if model_path.exists():
#     shutil.rmtree(model_path)
#     print(f"✅ Removed model directory: {model_path}")
# else:
#     print(f"ℹ Model directory not found: {model_path}")